# Parts B & C: implicit-scenario and numerical prompt probes (Qwen3-8B)

DSC 180A · Member 2 (Validation and generalization) · Sofroniew et al. (2026) §1.2 on Qwen3-8B.

> **Runs on Google Colab with an NVIDIA GPU** (a T4 is enough; NF4 needs CUDA).
> `DRY_RUN = True` swaps in **Qwen3-0.6B on CPU with random vectors. That is a CODE CHECK ONLY. Its numbers are NOT results.**

- **B (implicit scenarios):** do the emotion vectors pick out the intended emotion of a user message that never names it?
- **C (numerical):** as a number in the prompt makes the situation more or less dire, do the desperate, calm, afraid, sad and happy scores move with it? A number-only control (grapes) is included.

**Readout:** the output of decoder block 23 (the same hook point as notebook 03) at the **last prompt token** of the chat-formatted user message.
With `enable_thinking=False`, Qwen3's template ends in `<|im_start|>assistant\n<think>\n\n</think>\n\n`, so that last token is the `\n\n` after the empty think block. §4 prints it.

**Score:** `cos(h − neutral_baseline, unit_vectors[e, 23])`, where `neutral_baseline` is the mean readout over the neutral prompts.

**Prompts:** `prompts.json`. Status: **draft, awaiting review.**

### Colab setup (run once; skip locally)
```python
from google.colab import drive; drive.mount('/content/drive')
!git clone -b flavia/validation-generalization https://github.com/praveendesharma/dsc180a-steering-llms.git
%cd dsc180a-steering-llms/validation_generalization
```
Then set `RUN_DIR` below to the Drive folder holding `emotion_vectors.npz` (the unzipped archive).

## 1. Mode and dependencies
The install line is copied from notebook 03 and skipped in `DRY_RUN`.

In [ ]:
import os, sys
# True -> Qwen3-0.6B on CPU + random vectors: CODE CHECK ONLY, NOT RESULTS. Locally: BC_DRY_RUN=1.
DRY_RUN = os.environ.get("BC_DRY_RUN", "0") == "1"
if not DRY_RUN:
    %pip -q install "transformers>=4.51.0,<5" "accelerate>=1.6,<2" "bitsandbytes>=0.45,<1" safetensors numpy pandas matplotlib tqdm scipy

## 2. Configuration
`RUN_DIR` is the only path to change if the Drive archive lives somewhere else.

In [ ]:
import hashlib, json, re, time, zipfile
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from scipy.stats import spearmanr
from tqdm.auto import tqdm

# Folder holding emotion_vectors.npz (the unzipped Drive archive), relative to this notebook.
# On Colab, e.g. Path("/content/drive/MyDrive/<your folder>/output_drive").
RUN_DIR = Path("../emotion_vector_pipeline/output_drive")
DATASET_ZIP = Path("../emotion_vector_pipeline/data/qwen3_8b_emotion_stories_1120_v1.zip")   # forbidden-term lists
PROMPTS_PATH = Path("prompts.json")

MODEL_ID = "Qwen/Qwen3-8B"
MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"   # extraction run ab1333f49024705c
MAIN_LAYER = 23
C_EMOTIONS = ["desperate", "calm", "afraid", "sad", "happy"]
SEED = 180

if DRY_RUN:
    MODEL_ID, MODEL_REVISION = "Qwen/Qwen3-0.6B", None      # any cached revision; code check only
    DEVICE = torch.device("cpu")
    OUT_DIR = Path("outputs/BC_DRYRUN_code_check_only")
    LABEL = "CODE CHECK ONLY: not results (Qwen3-0.6B, CPU, random vectors)"
else:
    DEVICE = torch.device("cuda:0")
    if not torch.cuda.is_available():
        raise RuntimeError("This NF4 configuration requires an NVIDIA CUDA GPU (use Colab), "
                           "or set BC_DRY_RUN=1 for the CPU code check.")
    print("GPU:", torch.cuda.get_device_name(DEVICE))
    OUT_DIR = Path("outputs/BC")
    LABEL = f"Qwen3-8B NF4, block {MAIN_LAYER}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)

INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
SERIES = {"desperate": "#2a78d6", "calm": "#eb6834", "afraid": "#1baf7a", "sad": "#eda100", "happy": "#e87ba4"}
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#184f95", "#6da7ec", "#f0efec", "#ec8a89", "#b3302f"])
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "font.size": 10, "axes.titlesize": 11, "legend.frameon": False, "figure.dpi": 110,
})
print("DRY_RUN:", DRY_RUN, "|", LABEL, "| OUT_DIR:", OUT_DIR)

## 3. Prompts and the forbidden-term check
Every B and neutral prompt, and every C template with each value filled in, is checked against all 14 manifest term lists plus `extra_forbidden_terms`.
Matching is case-insensitive, on whole words, and allows suffixes -s/-es/-d/-ed/-ly/-ing/-ness. Any hit stops the notebook.

In [ ]:
prompts = json.loads(PROMPTS_PATH.read_text(encoding="utf-8"))
print("prompts.json status:", prompts.get("status"))
with zipfile.ZipFile(DATASET_ZIP) as zf:
    manifest_terms = json.loads(zf.read("manifest.json"))["config"]["terms"]
extra_terms = {k: v for k, v in prompts.get("extra_forbidden_terms", {}).items() if not k.startswith("_")}
forbidden = sorted({t.lower() for terms in [*manifest_terms.values(), *extra_terms.values()] for t in terms})
term_re = re.compile(r"\b(" + "|".join(map(re.escape, forbidden)) + r")(?:s|es|d|ed|ly|ing|ness)?\b", re.IGNORECASE)

def forbidden_hits(text):
    return sorted({m.group(0).lower() for m in term_re.finditer(text)})

# The checker itself must catch inflections and must not flag look-alikes.
assert forbidden_hits("She was Calmly sipping; we Loved it; sadness.") == ["calmly", "loved", "sadness"]
assert forbidden_hits("I made an average cake with a joystick.") == []

B, NEUTRAL, C = prompts["B_implicit"], prompts["neutral"], prompts["C_numerical"]
emotions_14 = sorted(manifest_terms)
to_check = [(p["id"], p["prompt"]) for p in B + NEUTRAL]
to_check += [(f'{t["id"]}[X={x}]', t["template"].format(X=x)) for t in C for x in t["values"]]
violations = {pid: hits for pid, text in to_check if (hits := forbidden_hits(text))}

ids = [p["id"] for p in B + NEUTRAL] + [t["id"] for t in C]
assert len(ids) == len(set(ids)), "duplicate prompt ids"
assert all(p["emotion"] in emotions_14 for p in B)
per_emotion = {e: sum(p["emotion"] == e for p in B) for e in emotions_14}
assert all(2 <= n <= 3 for n in per_emotion.values()), per_emotion
assert all("{X}" in t["template"] and 8 <= len(t["values"]) <= 10 for t in C)
print(f"{len(forbidden)} forbidden terms | checked {len(to_check)} texts "
      f"({len(B)} B, {len(NEUTRAL)} neutral, {len(to_check) - len(B) - len(NEUTRAL)} C)")
print("B scenarios per emotion:", per_emotion)
if violations:
    raise ValueError(f"Forbidden emotion terms found: {violations}")
print("No forbidden terms found.")

## 4. Vectors and model
The model-loading code is copied from notebook 03 (NF4, double quant, fp16 compute, sdpa) with the revision pinned to the extraction run.

In [ ]:
VEC_PATH = RUN_DIR / "emotion_vectors.npz"
if VEC_PATH.is_file():
    with np.load(VEC_PATH, allow_pickle=False) as z:
        emotion_names = z["emotion_names"].astype(str).tolist()     # read the order, never assume it
        layer_indices = z["layer_indices"].tolist()
        unit_main = z["unit_vectors"][:, layer_indices.index(MAIN_LAYER)].astype(np.float32)   # (E, D)
    print("vectors:", VEC_PATH, unit_main.shape, "| order:", emotion_names)
elif DRY_RUN:
    emotion_names, unit_main = emotions_14, None
    print("emotion_vectors.npz not found; fine for DRY_RUN (random vectors are used anyway).")
else:
    raise FileNotFoundError(f"No vectors at {VEC_PATH.resolve()}. Set RUN_DIR to the unzipped Drive archive.")
assert set(emotion_names) == set(emotions_14)
E = len(emotion_names)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

if DRY_RUN:
    from huggingface_hub import try_to_load_from_cache
    cached = try_to_load_from_cache(MODEL_ID, "config.json")        # .../snapshots/<commit>/config.json
    resolved_revision = Path(cached).parent.name if isinstance(cached, str) else "main"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=resolved_revision)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=resolved_revision).to(DEVICE, torch.float32)
else:
    from huggingface_hub import HfApi
    from transformers import BitsAndBytesConfig
    resolved_revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
    assert resolved_revision == MODEL_REVISION, resolved_revision
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=resolved_revision)
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=resolved_revision,
        quantization_config=quant_config,
        torch_dtype=torch.float16,
        device_map={'': str(DEVICE)},
        low_cpu_mem_usage=True,
        attn_implementation='sdpa',
    )
model.eval()
model.config.use_cache = False
blocks = model.model.layers
HIDDEN = model.config.hidden_size
assert 0 <= MAIN_LAYER < len(blocks)
print(f"{MODEL_ID} @ {resolved_revision} | blocks {len(blocks)} | hidden {HIDDEN} | "
      f"transformers {transformers.__version__} | torch {torch.__version__}")

if DRY_RUN:
    rng = np.random.default_rng(SEED)
    unit_main = rng.standard_normal((E, HIDDEN)).astype(np.float32)
    unit_main /= np.linalg.norm(unit_main, axis=1, keepdims=True)
    print("DRY_RUN: random unit vectors", unit_main.shape, "(CODE CHECK ONLY)")
assert unit_main.shape == (E, HIDDEN), (unit_main.shape, HIDDEN)

## 5. Prompt building, readout hook, and tests
Tests: the template ends with the empty think block; the hook returns shape `(hidden,)`; the hook only reads (outputs are unchanged while it is attached and after it is removed); the hook really is removed; and the hooked state equals `hidden_states[MAIN_LAYER + 1]`.

In [ ]:
def build_prompt(p):
    return tokenizer.apply_chat_template([{"role": "user", "content": p}], add_generation_prompt=True,
                                         enable_thinking=False, tokenize=False)

def encode(p):
    # The template already holds the special tokens, so add none.
    return tokenizer(build_prompt(p), add_special_tokens=False, return_tensors="pt")["input_ids"].to(DEVICE)

def last_token_state(p, layer=MAIN_LAYER):
    # Block `layer` output (notebook 03's hook point) at the final prompt token -> float32 (hidden,).
    ids = encode(p)
    captured = {}
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        assert hidden.ndim == 3 and hidden.shape[0] == 1
        captured["h"] = hidden[0, -1].float().cpu().numpy()
    handle = blocks[layer].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            model.model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False, return_dict=True)
    finally:
        handle.remove()
    return captured["h"]

example = build_prompt(B[0]["prompt"])
ex_ids = encode(B[0]["prompt"])
print(repr(example))
print("tokens:", ex_ids.shape[1], "| last prompt token:", repr(tokenizer.decode(ex_ids[0, -1:])))
assert example.endswith("<|im_start|>assistant\n<think>\n\n</think>\n\n"), "unexpected chat-template ending"
assert tokenizer.decode(ex_ids[0]) == example, "tokenization does not round-trip"

In [ ]:
probe = NEUTRAL[0]["prompt"]
ids = encode(probe)
run = lambda: model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits.float().cpu()
with torch.inference_mode():
    logits_before = run()
    ref_state = model.model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False,
                            output_hidden_states=True).hidden_states[MAIN_LAYER + 1][0, -1].float().cpu().numpy()

# transformers 5.x leaves its own output-capture hook on each block after output_hidden_states=True,
# so "removed" means: our hook id is gone and the block's hook set is exactly what it was before.
hooks_before = set(blocks[MAIN_LAYER]._forward_hooks)
captured = {}
def _probe_hook(module, args, output):
    captured["h"] = (output[0] if isinstance(output, tuple) else output)[0, -1].float().cpu().numpy()
handle = blocks[MAIN_LAYER].register_forward_hook(_probe_hook)
our_hook_id = handle.id
assert our_hook_id in blocks[MAIN_LAYER]._forward_hooks
try:
    with torch.inference_mode():
        logits_during = run()
finally:
    handle.remove()
with torch.inference_mode():
    logits_after = run()

state = last_token_state(probe)
tol = dict(rtol=1e-5, atol=1e-5) if DRY_RUN else dict(rtol=1e-3, atol=1e-3)   # notebook 03 uses 1e-3 on GPU
assert state.shape == (HIDDEN,), state.shape
assert np.isfinite(state).all()
assert our_hook_id not in blocks[MAIN_LAYER]._forward_hooks, "hook was not removed"
assert set(blocks[MAIN_LAYER]._forward_hooks) == hooks_before, "hook set changed after last_token_state"
torch.testing.assert_close(logits_during, logits_before, **tol)
torch.testing.assert_close(logits_after, logits_before, **tol)
np.testing.assert_allclose(state, captured["h"], **tol)
np.testing.assert_allclose(state, ref_state, **tol)
hook_tests = {
    "state_shape": list(state.shape),
    "max_abs_logit_diff_with_hook": float((logits_during - logits_before).abs().max()),
    "max_abs_logit_diff_after_removal": float((logits_after - logits_before).abs().max()),
    "max_abs_diff_vs_hidden_states": float(np.abs(state - ref_state).max()),
    "our_hook_removed": our_hook_id not in blocks[MAIN_LAYER]._forward_hooks,
    "other_library_hooks_on_block": len(hooks_before),
}
print("hook tests passed:", hook_tests)

## 6. Neutral baseline and score

In [ ]:
neutral_states = np.stack([last_token_state(p["prompt"]) for p in tqdm(NEUTRAL, desc="neutral")])
neutral_baseline = neutral_states.mean(axis=0)
assert neutral_states.shape == (len(NEUTRAL), HIDDEN)

def cos_scores(h):
    # cos(h - neutral_baseline, unit_vectors[e, MAIN_LAYER]) for every emotion e -> (E,)
    d = h - neutral_baseline
    return (unit_main @ d) / (np.linalg.norm(unit_main, axis=1) * np.linalg.norm(d))

neutral_scores = np.stack([cos_scores(h) for h in neutral_states])   # reference spread only
print("neutral prompts' own scores: |cos| max", round(float(np.abs(neutral_scores).max()), 3))

## 7. Part B: implicit scenarios
Rank 1 means the intended emotion's vector scored highest of the 14. Chance levels: top-1 = 1/14, top-3 = 3/14, mean rank = 7.5.

In [ ]:
B_states = np.stack([last_token_state(p["prompt"]) for p in tqdm(B, desc="B")])
B_scores = np.stack([cos_scores(h) for h in B_states])                  # (n_B, E)
intended = np.array([emotion_names.index(p["emotion"]) for p in B])
own = B_scores[np.arange(len(B)), intended]
ranks = (B_scores > own[:, None]).sum(axis=1) + 1
assert B_scores.shape == (len(B), E) and np.isfinite(B_scores).all()

B_summary = {
    "label": LABEL, "n_scenarios": len(B),
    "top1_hit_rate": float((ranks == 1).mean()), "top3_hit_rate": float((ranks <= 3).mean()),
    "mean_rank_of_intended": float(ranks.mean()),
    "chance": {"top1": 1 / E, "top3": 3 / E, "mean_rank": (E + 1) / 2},
    "per_emotion_mean_rank": {e: float(ranks[intended == k].mean()) for k, e in enumerate(emotion_names) if (intended == k).any()},
}
print(json.dumps(B_summary, indent=1))

In [ ]:
import csv
order = np.lexsort((np.arange(len(B)), intended))                      # group rows by intended emotion
with open(OUT_DIR / "B_scores.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "intended", "rank_of_intended"] + emotion_names)
    for i in order:
        w.writerow([B[i]["id"], B[i]["emotion"], int(ranks[i])] + [round(float(v), 5) for v in B_scores[i]])
    for i, p in enumerate(NEUTRAL):
        w.writerow([p["id"], "neutral", ""] + [round(float(v), 5) for v in neutral_scores[i]])
(OUT_DIR / "B_summary.json").write_text(json.dumps(B_summary, indent=2))

fig, ax = plt.subplots(figsize=(10, 0.28 * len(B) + 2))
vmax = float(np.abs(B_scores).max())
im = ax.imshow(B_scores[order], cmap=DIVERGING, vmin=-vmax, vmax=vmax, aspect="auto")
for r, i in enumerate(order):
    ax.add_patch(Rectangle((intended[i] - 0.5, r - 0.5), 1, 1, fill=False, ec=INK, lw=1.2))
ax.set_xticks(range(E), emotion_names, rotation=60, ha="right")
ax.set_yticks(range(len(B)), [f'{B[i]["id"]} (rank {ranks[i]})' for i in order], fontsize=8)
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="cos(h − neutral, vector)")
ax.set_title(f"Part B: implicit scenarios × emotion vectors (outlined = intended)\n{LABEL} | "
             f"top-1 {B_summary['top1_hit_rate']:.2f}, top-3 {B_summary['top3_hit_rate']:.2f}, "
             f"mean rank {B_summary['mean_rank_of_intended']:.1f}", fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "B_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

## 8. Part C: numerical templates
X is plotted against the desperate, calm, afraid, sad and happy scores, one panel per template, with a Spearman ρ for each line.
With 8–10 points per line, ρ is coarse. Read it alongside the plot and the grapes control.

In [ ]:
C_rows, C_rho = [], []
C_scores = {}
for t in tqdm(C, desc="C"):
    S = np.stack([cos_scores(last_token_state(t["template"].format(X=x))) for x in t["values"]])   # (n_X, E)
    assert S.shape == (len(t["values"]), E)
    C_scores[t["id"]] = S
    for x, row in zip(t["values"], S):
        C_rows.append([t["id"], x, t["template"].format(X=x)] + [round(float(v), 5) for v in row])
    for e in C_EMOTIONS:
        rho, p = spearmanr(t["values"], S[:, emotion_names.index(e)])
        C_rho.append({"template": t["id"], "emotion": e, "spearman_rho": round(float(rho), 4),
                      "p": round(float(p), 4), "n": len(t["values"])})

with open(OUT_DIR / "C_scores.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["template", "X", "prompt"] + emotion_names); w.writerows(C_rows)
with open(OUT_DIR / "C_spearman.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(C_rho[0])); w.writeheader(); w.writerows(C_rho)

ncol = 3
nrow = int(np.ceil(len(C) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.8 * nrow), squeeze=False)
for ax, t in zip(axes.flat, C):
    S, xs = C_scores[t["id"]], np.array(t["values"], float)
    rhos = []
    for e in C_EMOTIONS:
        rho = next(r["spearman_rho"] for r in C_rho if r["template"] == t["id"] and r["emotion"] == e)
        ax.plot(xs, S[:, emotion_names.index(e)], "-o", ms=4, color=SERIES[e], label=e)
        rhos.append(f"{e} {rho:+.2f}")
    ax.set_xscale("log")
    ax.axhline(0, color=MUTED, lw=1, ls=":")
    ax.set_title(t["template"].replace("{X}", "X") + "\nSpearman ρ: " + " · ".join(rhos), fontsize=8.5)
    ax.set_xlabel("X (log scale)"); ax.set_ylabel("cos score")
for ax in list(axes.flat)[len(C):]:
    ax.set_visible(False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(C_EMOTIONS), fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"Part C: numerical templates, {LABEL}", fontsize=11)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(OUT_DIR / "C_lines.png", dpi=150, bbox_inches="tight"); plt.show()
print("\n".join(f'{r["template"]:22s} {r["emotion"]:10s} rho={r["spearman_rho"]:+.2f} p={r["p"]:.3f}' for r in C_rho))

## 9. Run record

In [ ]:
run_info = {
    "label": LABEL, "dry_run": DRY_RUN, "model_id": MODEL_ID, "model_revision": resolved_revision,
    "main_layer": MAIN_LAYER, "device": str(DEVICE),
    "transformers": transformers.__version__, "torch": torch.__version__,
    "prompts_sha256": hashlib.sha256(PROMPTS_PATH.read_bytes()).hexdigest(),
    "prompts_status": prompts.get("status"),
    "n_prompts": {"B": len(B), "neutral": len(NEUTRAL), "C": sum(len(t["values"]) for t in C)},
    "hook_tests": hook_tests, "emotion_names": emotion_names,
    "vectors": "random unit vectors (dry run)" if DRY_RUN else str(VEC_PATH),
}
(OUT_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(json.dumps(run_info, indent=1))
print("\nOutputs in", OUT_DIR.resolve())
if DRY_RUN:
    print("\n*** DRY RUN: CODE CHECK ONLY. These numbers are NOT results. ***")